In [ ]:
# ============================================================
# İSTANBUL 39 İLÇE SU TÜKETİM TAHMİNİ - RANDOM FOREST
# Konut ve işyeri ayrı ayrı tahmin edilir, sonra toplanır
# Eğitim: 2020-01 → 2024-12  |  Tahmin: 2025-01 → 2025-12
#
# Random Forest yaklaşımı:
#   Özellikler: ay, yıl, sıcaklık, konut_abone, isyeri_abone
#   Strateji  : tüm ilçeler tek modelde ilce_id ile temsil edilir
#   Parametre : n_estimators=100, max_depth=8, random_state=42
#   Avantaj   : Karar Ağacı'na göre daha az aşırı öğrenme,
#               bootstrap örnekleme ile varyans azaltımı
# ============================================================

# ── AŞAMA 1: KÜTÜPHANELERİ İÇE AKTAR ──────────────────────
# Kurulum: pip install scikit-learn openpyxl

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error
import sklearn

print("✅ Kütüphaneler yüklendi")
print(f"   scikit-learn: {sklearn.__version__}")

In [ ]:
# ── AŞAMA 2: DOSYA YOLLARI ─────────────────────────────────
ABONE_DOSYA    = r"abone.xlsx"
SU_DOSYA       = r"su_tuketim.xlsx"
SICAKLIK_DOSYA = r"Sıcaklık_Verileri.xlsx"

In [ ]:
# ── AŞAMA 3: YARDIMCI FONKSİYON - TARİH PARSE ─────────────
AYLAR_TR = {
    "Ocak":1, "Şubat":2, "Mart":3, "Nisan":4,
    "Mayıs":5, "Haziran":6, "Temmuz":7, "Ağustos":8,
    "Eylül":9, "Ekim":10, "Kasım":11, "Aralık":12
}

def tarih_parse(tarih_str):
    parcalar = str(tarih_str).strip().split()
    yil = int(parcalar[0])
    ay  = AYLAR_TR[parcalar[-1]]
    return pd.Timestamp(year=yil, month=ay, day=1)

In [ ]:
# ── AŞAMA 4: ABONE VERİSİNİ YÜKLE ──────────────────────────
abone_raw = pd.read_excel(ABONE_DOSYA, header=1)

konut = abone_raw.iloc[:, 0:3].copy()
konut.columns = ["ilce", "tarih", "konut_abone"]

isyeri = abone_raw.iloc[:, 8:11].copy()
isyeri.columns = ["ilce", "tarih", "isyeri_abone"]

konut  = konut.dropna(subset=["ilce", "tarih"]).reset_index(drop=True)
isyeri = isyeri.dropna(subset=["ilce", "tarih"]).reset_index(drop=True)

konut["tarih"]  = konut["tarih"].apply(tarih_parse)
isyeri["tarih"] = isyeri["tarih"].apply(tarih_parse)

abone_df = pd.merge(konut, isyeri, on=["ilce", "tarih"], how="outer")
abone_df = abone_df[["ilce", "tarih", "konut_abone", "isyeri_abone"]]

print(f"✅ Abone verisi: {abone_df.shape[0]} satır, {abone_df['ilce'].nunique()} ilçe")

In [ ]:
# ── AŞAMA 5: SU TÜKETİM VERİSİNİ YÜKLE ────────────────────
su_raw = pd.read_excel(SU_DOSYA, header=1)

konut_su = su_raw.iloc[:, 0:3].copy()
konut_su.columns = ["ilce", "tarih", "konut_m3"]

isyeri_su = su_raw.iloc[:, 4:7].copy()
isyeri_su.columns = ["ilce", "tarih", "isyeri_m3"]

konut_su  = konut_su.dropna(subset=["ilce", "tarih"]).reset_index(drop=True)
isyeri_su = isyeri_su.dropna(subset=["ilce", "tarih"]).reset_index(drop=True)

konut_su["tarih"]  = konut_su["tarih"].apply(tarih_parse)
isyeri_su["tarih"] = isyeri_su["tarih"].apply(tarih_parse)

konut_su["konut_m3"]   = pd.to_numeric(konut_su["konut_m3"],   errors="coerce")
isyeri_su["isyeri_m3"] = pd.to_numeric(isyeri_su["isyeri_m3"], errors="coerce")

su_df = pd.merge(konut_su, isyeri_su, on=["ilce", "tarih"], how="outer")
su_df = su_df[["ilce", "tarih", "konut_m3", "isyeri_m3"]]

print(f"✅ Su tüketim verisi: {su_df.shape[0]} satır, {su_df['ilce'].nunique()} ilçe")

In [ ]:
# ── AŞAMA 6: SICAKLIK VERİSİNİ YÜKLE ───────────────────────
sicaklik_raw = pd.read_excel(SICAKLIK_DOSYA, header=1)

sicaklik_df = sicaklik_raw.rename(columns={
    "İLÇE": "ilce",
    "TARİH": "tarih",
    "ORTALAMA SICAKLIK": "sicaklik_c"
})

tarih_split = sicaklik_df["tarih"].astype(str).str.strip().str.split(r"\s+", expand=True)
sicaklik_df["tarih"] = pd.to_datetime(
    tarih_split[0] + "-" + tarih_split[1].map(AYLAR_TR).astype(str),
    format="%Y-%m"
)

sicaklik_df["sicaklik_c"] = (
    sicaklik_df["sicaklik_c"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

sicaklik_df["ilce"] = sicaklik_df["ilce"].str.upper().str.strip()

print(f"✅ Sıcaklık verisi: {sicaklik_df.shape[0]} satır, {sicaklik_df['ilce'].nunique()} ilçe")

In [ ]:
# ── AŞAMA 7: İLÇE İSİMLERİNİ EŞLEŞTİR ─────────────────────
ilce_su       = set(su_df["ilce"].str.upper().str.strip().unique())
ilce_abone    = set(abone_df["ilce"].str.upper().str.strip().unique())
ilce_sicaklik = set(sicaklik_df["ilce"].unique())

print(f"Su verisi      : {len(ilce_su)} ilçe")
print(f"Abone verisi   : {len(ilce_abone)} ilçe")
print(f"Sıcaklık verisi: {len(ilce_sicaklik)} ilçe")

print("\n── Su ∩ Abone ──────────────────────────────")
print("🔴 Abonede olup suda OLMAYAN  :", ilce_abone - ilce_su)
print("🔵 Suda olup abonede OLMAYAN  :", ilce_su - ilce_abone)

print("\n── Su ∩ Sıcaklık ───────────────────────────")
print("🔴 Sıcaklıkta olup suda OLMAYAN  :", ilce_sicaklik - ilce_su)
print("🔵 Suda olup sıcaklıkta OLMAYAN  :", ilce_su - ilce_sicaklik)

print("\n── Üç sette birden olan ilçe sayısı ────────")
print("✅", len(ilce_su & ilce_abone & ilce_sicaklik), "ilçe tam eşleşiyor")

tum_ilceler = ilce_su | ilce_abone | ilce_sicaklik
eksik = {i: [] for i in tum_ilceler}
for i in tum_ilceler:
    if i not in ilce_su:       eksik[i].append("su yok")
    if i not in ilce_abone:    eksik[i].append("abone yok")
    if i not in ilce_sicaklik: eksik[i].append("sıcaklık yok")
eksik = {k: v for k, v in eksik.items() if v}
for ilce, sorun in sorted(eksik.items()):
    print(f"  {ilce:25s} → {', '.join(sorun)}")

In [ ]:
# ── AŞAMA 8: ÜÇ VERİYİ BİRLEŞTİR ──────────────────────────
df = (
    su_df
    .merge(abone_df,    on=["ilce", "tarih"], how="inner")
    .merge(sicaklik_df, on=["ilce", "tarih"], how="inner")
)
df = df[["ilce", "tarih", "konut_m3", "isyeri_m3",
         "konut_abone", "isyeri_abone", "sicaklik_c"]]
df["yil"] = df["tarih"].dt.year
df["ay"]  = df["tarih"].dt.month
df["su_tuketimi_m3"] = df["konut_m3"] + df["isyeri_m3"]

print(f"✅ Birleşik veri: {df.shape[0]} satır, {df['ilce'].nunique()} ilçe")
print(f"📅 Tarih aralığı: {df['tarih'].min().strftime('%Y-%m')} → {df['tarih'].max().strftime('%Y-%m')}")

In [ ]:
# ── AŞAMA 9: KEŞİFÇİ ANALİZ ─────────────────────────────────
print("── Eksik değer kontrolü ──")
print(df.isnull().sum())

print("\n── Temel istatistikler ──")
print(df[["konut_m3", "isyeri_m3", "su_tuketimi_m3", "sicaklik_c"]].describe().round(0))

print("\n── İlçe başına ortalama aylık tüketim (Top 5) ──")
ilce_ort = df.groupby("ilce")["su_tuketimi_m3"].mean().sort_values(ascending=False)
print(ilce_ort.head())

In [ ]:
# ── AŞAMA 10: KEŞİFÇİ GRAFİKLER ───────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("İstanbul Su Tüketim Analizi (2020-2025)", fontsize=14, fontweight="bold")

# 1. Yıllık toplam
ax1 = axes[0, 0]
yillik_tam = df[df["yil"] < 2025].groupby("yil")["su_tuketimi_m3"].sum() / 1e6
ax1.bar(yillik_tam.index, yillik_tam.values, color="#1a6fae", alpha=0.85)
ax1.set_title("Yıllık Toplam Tüketim (2020-2024)")
ax1.set_ylabel("Milyon m³")
ax1.set_xticks(yillik_tam.index)
for i, v in enumerate(yillik_tam.values):
    ax1.text(yillik_tam.index[i], v + 1, f"{v:.0f}M", ha="center", fontsize=9)

# 2. Konut vs işyeri
ax2 = axes[0, 1]
yillik_konut  = df[df["yil"] < 2025].groupby("yil")["konut_m3"].sum() / 1e6
yillik_isyeri = df[df["yil"] < 2025].groupby("yil")["isyeri_m3"].sum() / 1e6
x = np.arange(len(yillik_konut))
ax2.bar(x - 0.2, yillik_konut.values, width=0.4, label="Konut", color="#1a6fae", alpha=0.85)
ax2.bar(x + 0.2, yillik_isyeri.values, width=0.4, label="İşyeri", color="#e05c1a", alpha=0.85)
ax2.set_title("Konut vs İşyeri Tüketimi")
ax2.set_ylabel("Milyon m³")
ax2.set_xticks(x)
ax2.set_xticklabels(yillik_konut.index)
ax2.legend()

# 3. İlçeler
ax3 = axes[0, 2]
top10 = ilce_ort.head(10) / 1e6
ax3.barh(top10.index[::-1], top10.values[::-1], color="#e05c1a", alpha=0.85)
ax3.set_title("En Çok Tüketen 10 İlçe")
ax3.set_xlabel("Milyon m³")

# 4. Mevsimsellik
ax4 = axes[1, 0]
aylik_konut  = df.groupby("ay")["konut_m3"].mean() / 1e6
aylik_isyeri = df.groupby("ay")["isyeri_m3"].mean() / 1e6
ay_isimleri = ["Oca","Şub","Mar","Nis","May","Haz","Tem","Ağu","Eyl","Eki","Kas","Ara"]
ax4.plot(aylik_konut.index, aylik_konut.values, color="#1a6fae", marker="o", label="Konut")
ax4.plot(aylik_isyeri.index, aylik_isyeri.values, color="#e05c1a", marker="s", label="İşyeri")
ax4.set_title("Mevsimsel Patern")
ax4.set_xticks(range(1, 13))
ax4.set_xticklabels(ay_isimleri)
ax4.legend()
ax4.grid(True, alpha=0.3)

# 5. Sıcaklık - Konut
ax5 = axes[1, 1]
x1 = pd.to_numeric(df["sicaklik_c"], errors="coerce")
y1 = pd.to_numeric(df["konut_m3"], errors="coerce") / 1e6
mask1 = np.isfinite(x1) & np.isfinite(y1)
x1, y1 = x1[mask1], y1[mask1]
ax5.scatter(x1, y1, alpha=0.15, color="#1a6fae", s=10)
ax5.set_title("Sıcaklık - Konut Tüketimi")
ax5.set_xlabel("Sıcaklık (°C)")
ax5.set_ylabel("Milyon m³")
if len(x1) > 1:
    z = np.polyfit(x1, y1, 1)
    xp = np.linspace(x1.min(), x1.max(), 100)
    ax5.plot(xp, np.poly1d(z)(xp), color="red", linewidth=2)

# 6. Sıcaklık - İşyeri
ax6 = axes[1, 2]
x2 = pd.to_numeric(df["sicaklik_c"], errors="coerce")
y2 = pd.to_numeric(df["isyeri_m3"], errors="coerce") / 1e6
mask2 = np.isfinite(x2) & np.isfinite(y2)
x2, y2 = x2[mask2], y2[mask2]
ax6.scatter(x2, y2, alpha=0.15, color="#e05c1a", s=10)
ax6.set_title("Sıcaklık - İşyeri Tüketimi")
ax6.set_xlabel("Sıcaklık (°C)")
ax6.set_ylabel("Milyon m³")
if len(x2) > 1:
    z2 = np.polyfit(x2, y2, 1)
    xp2 = np.linspace(x2.min(), x2.max(), 100)
    ax6.plot(xp2, np.poly1d(z2)(xp2), color="red", linewidth=2)

plt.tight_layout()
plt.savefig("analiz_rf.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Grafik kaydedildi: analiz_rf.png")

In [ ]:
# ── AŞAMA 11: EĞİTİM / VALİDASYON AYRIMI ───────────────────
# Eğitim: 2020-01 → 2024-12
# Validasyon (test): 2025-01 → 2025-12

EGITIM_BITIS = "2024-12-01"
FEATURES     = ["ay", "yil", "sicaklik_c", "konut_abone", "isyeri_abone", "ilce_id"]

# İlçe label encoding (tüm modelde ortak)
le = LabelEncoder()
df["ilce_id"] = le.fit_transform(df["ilce"].str.upper().str.strip())

train_df = df[df["tarih"] <= EGITIM_BITIS].copy()
test_df  = df[df["tarih"] >  EGITIM_BITIS].copy()

print(f"✅ Eğitim    : {train_df['tarih'].min().strftime('%Y-%m')} → {train_df['tarih'].max().strftime('%Y-%m')}  ({len(train_df)} satır)")
print(f"✅ Validasyon: {test_df['tarih'].min().strftime('%Y-%m')}  → {test_df['tarih'].max().strftime('%Y-%m')}   ({len(test_df)} satır)")

In [ ]:
# ── AŞAMA 12: RANDOM FOREST - KONUT MODELİ ──────────────────
# n_estimators=100 : 100 karar ağacı, toplulaştırarak varyansı düşürür
# max_depth=8      : her ağaç maksimum 8 seviye derine iner
# n_jobs=-1        : tüm CPU çekirdeklerini kullan (hız)
# random_state=42  : tekrarlanabilir sonuçlar

X_train_k = train_df[FEATURES]
y_train_k = train_df["konut_m3"]
X_test_k  = test_df[FEATURES]

rf_konut = RandomForestRegressor(
    n_estimators=100,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)
rf_konut.fit(X_train_k, y_train_k)

test_df = test_df.copy()
test_df["konut_tahmin"] = rf_konut.predict(X_test_k)

print("✅ Random Forest Konut modeli eğitildi")
print(f"   Ağaç sayısı    : {rf_konut.n_estimators}")
print(f"   Özellik sayısı : {len(FEATURES)}")
print(f"   Max derinlik   : {rf_konut.max_depth}")

In [ ]:
# ── AŞAMA 13: RANDOM FOREST - İŞYERİ MODELİ ────────────────
X_train_i = train_df[FEATURES]
y_train_i = train_df["isyeri_m3"]
X_test_i  = test_df[FEATURES]

rf_isyeri = RandomForestRegressor(
    n_estimators=100,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)
rf_isyeri.fit(X_train_i, y_train_i)

test_df["isyeri_tahmin"] = rf_isyeri.predict(X_test_i)

print("✅ Random Forest İşyeri modeli eğitildi")
print(f"   Ağaç sayısı    : {rf_isyeri.n_estimators}")
print(f"   Özellik sayısı : {len(FEATURES)}")
print(f"   Max derinlik   : {rf_isyeri.max_depth}")

In [ ]:
# ── AŞAMA 14: TOPLAM TAHMİN HESAPLA ─────────────────────────
test_df["toplam_tahmin"] = test_df["konut_tahmin"] + test_df["isyeri_tahmin"]
test_df["toplam_gercek"] = test_df["konut_m3"]    + test_df["isyeri_m3"]

print("✅ Toplam tahmin hesaplandı (konut + işyeri)")
print(f"   Test satır sayısı: {len(test_df)}")
print(test_df[["ilce", "tarih", "konut_tahmin", "isyeri_tahmin", "toplam_tahmin", "toplam_gercek"]].head(10).to_string(index=False))

In [ ]:
# ── AŞAMA 15: KARŞILAŞTIRMA TABLOSU ─────────────────────────
karsilastirma = test_df[[
    "ilce", "tarih",
    "konut_m3",     "konut_tahmin",
    "isyeri_m3",    "isyeri_tahmin",
    "toplam_gercek","toplam_tahmin"
]].copy()

karsilastirma.columns = [
    "unique_id", "ds",
    "konut_gercek",  "konut_tahmin",
    "isyeri_gercek", "isyeri_tahmin",
    "toplam_gercek", "toplam_tahmin"
]

print(f"✅ Karşılaştırma tablosu hazır: {karsilastirma.shape}")
print(karsilastirma.head(5).to_string(index=False))

In [ ]:
# ── AŞAMA 16: PERFORMANS METRİKLERİ ─────────────────────────
# MAPE, MAE, RMSE — her ilçe için ayrı hesaplanır

metrik_satirlar = []

for ilce in karsilastirma["unique_id"].unique():
    grp = karsilastirma[karsilastirma["unique_id"] == ilce]
    satir = {"unique_id": ilce}

    for hedef, gercek_col, tahmin_col in [
        ("Konut",  "konut_gercek",  "konut_tahmin"),
        ("Isyeri", "isyeri_gercek", "isyeri_tahmin"),
        ("Toplam", "toplam_gercek", "toplam_tahmin"),
    ]:
        y    = grp[gercek_col]
        yhat = grp[tahmin_col]
        mape = np.mean(np.abs((y - yhat) / (np.abs(y) + 1e-9))) * 100
        mae  = mean_absolute_error(y, yhat)
        rmse = np.sqrt(mean_squared_error(y, yhat))
        satir[f"MAPE_{hedef}"] = round(mape, 2)
        satir[f"MAE_{hedef}"]  = round(mae,  0)
        satir[f"RMSE_{hedef}"] = round(rmse, 0)

    metrik_satirlar.append(satir)

metrikler = pd.DataFrame(metrik_satirlar).sort_values("MAPE_Toplam").reset_index(drop=True)

print("✅ Performans metrikleri hesaplandı")
print(f"\n📊 En iyi 5 ilçe (MAPE_Toplam):")
print(metrikler[["unique_id","MAPE_Konut","MAPE_Isyeri","MAPE_Toplam"]].head(5).to_string(index=False))
print(f"\n📊 En kötü 5 ilçe (MAPE_Toplam):")
print(metrikler[["unique_id","MAPE_Konut","MAPE_Isyeri","MAPE_Toplam"]].tail(5).to_string(index=False))
print(f"\n📊 Genel ortalama MAPE (Toplam): {metrikler['MAPE_Toplam'].mean():.2f}%")

In [ ]:
# ── AŞAMA 17: ÖZELLİK ÖNEMİ GRAFİĞİ ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Random Forest — Özellik Önemi", fontsize=13, fontweight="bold")

for ax, model, baslik in [
    (axes[0], rf_konut,  "Konut Modeli"),
    (axes[1], rf_isyeri, "İşyeri Modeli")
]:
    imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=True)
    ax.barh(imp.index, imp.values, color="#1a6fae", alpha=0.85)
    ax.set_title(baslik)
    ax.set_xlabel("Önem Skoru")
    ax.grid(True, alpha=0.3, axis="x")

plt.tight_layout()
plt.savefig("ozellik_onemi_rf.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Özellik önemi grafiği kaydedildi: ozellik_onemi_rf.png")

In [ ]:
# ── AŞAMA 18: TAHMİN GRAFİKLERİ ─────────────────────────────
# En iyi 2 + En kötü 2 ilçe gösterilir

en_iyi  = metrikler.head(2)["unique_id"].tolist()
en_kotu = metrikler.tail(2)["unique_id"].tolist()
gosterilecek = en_iyi + en_kotu

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Random Forest Tahmin vs Gerçek — Toplam Tüketim", fontsize=14, fontweight="bold")

etiketler = ["EN İYİ 1", "EN İYİ 2", "EN KÖTÜ 1", "EN KÖTÜ 2"]

for ax, ilce, etiket in zip(axes.flatten(), gosterilecek, etiketler):
    # Geçmiş (eğitim) verisi
    gecmis = df[df["ilce"] == ilce].copy()
    gecmis = gecmis[gecmis["tarih"] <= EGITIM_BITIS]
    ax.plot(gecmis["tarih"], gecmis["su_tuketimi_m3"] / 1e6,
            color="steelblue", linewidth=1.5, label="Geçmiş (eğitim)")

    # Test dönemi
    k = karsilastirma[karsilastirma["unique_id"] == ilce]
    mape_val = metrikler[metrikler["unique_id"] == ilce]["MAPE_Toplam"].values[0]

    ax.plot(k["ds"], k["toplam_gercek"] / 1e6,
            color="black", linewidth=2, label="Gerçek")
    ax.plot(k["ds"], k["toplam_tahmin"] / 1e6,
            color="red", linewidth=2, linestyle="--", label="Tahmin (Random Forest)")

    ax.axvline(pd.Timestamp("2025-01-01"), color="gray", linestyle=":", linewidth=1)
    ax.set_title(f"{ilce}  |  MAPE: {mape_val:.1f}%")
    ax.set_ylabel("Milyon m³")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    badge_color = "#1A5276" if etiket.startswith("EN İYİ") else "#7B241C"
    ax.text(0.02, 0.95, etiket, transform=ax.transAxes, fontsize=8,
            fontweight="bold", color="white", va="top", ha="left",
            bbox=dict(boxstyle="round,pad=0.3", facecolor=badge_color, alpha=0.8))

plt.tight_layout()
plt.savefig("tahmin_grafikleri_rf.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Tahmin grafikleri kaydedildi: tahmin_grafikleri_rf.png")

In [ ]:
# ── AŞAMA 19: BAĞLAM VE TAHMİN TABLOLARINI HAZIRLA ──────────
# Excel çıktısı için gerekli ek tablolar

# Bağlam tabloları (eğitim verisi)
df_ctx_konut = train_df[["ilce", "tarih", "konut_m3", "sicaklik_c", "konut_abone", "isyeri_abone"]].copy()
df_ctx_konut.columns = ["unique_id", "ds", "y", "sicaklik_c", "konut_abone", "isyeri_abone"]

df_ctx_isyeri = train_df[["ilce", "tarih", "isyeri_m3", "sicaklik_c", "konut_abone", "isyeri_abone"]].copy()
df_ctx_isyeri.columns = ["unique_id", "ds", "y", "sicaklik_c", "konut_abone", "isyeri_abone"]

# Sadece tahminler (test dönemi)
tahminler = karsilastirma[["unique_id", "ds", "konut_tahmin", "isyeri_tahmin", "toplam_tahmin"]].copy()

# Ham veri (tüm dönem)
ham_veri = df[["ilce", "tarih", "konut_m3", "isyeri_m3", "konut_abone", "isyeri_abone", "sicaklik_c", "yil", "ay"]].copy()
ham_veri.columns = ["unique_id", "ds", "konut_m3", "isyeri_m3", "konut_abone", "isyeri_abone", "sicaklik_c", "yil", "ay"]

print(f"✅ Bağlam Konut  : {df_ctx_konut.shape}")
print(f"✅ Bağlam İşyeri : {df_ctx_isyeri.shape}")
print(f"✅ Tahminler     : {tahminler.shape}")
print(f"✅ Ham Veri      : {ham_veri.shape}")

In [ ]:
# ── AŞAMA 20: SONUÇLARI EXCEL'E KAYDET ─────────────────────
CIKTI_DOSYASI = "istanbul_su_tahmini_random_forest.xlsx"

with pd.ExcelWriter(CIKTI_DOSYASI, engine="openpyxl") as writer:
    karsilastirma.to_excel(writer, sheet_name="Tahmin vs Gerçek",  index=False)
    metrikler.to_excel(    writer, sheet_name="Performans",         index=False)
    tahminler.to_excel(    writer, sheet_name="Tahminler",          index=False)
    df_ctx_konut.to_excel( writer, sheet_name="Bağlam Konut",      index=False)
    df_ctx_isyeri.to_excel(writer, sheet_name="Bağlam İşyeri",     index=False)
    ham_veri.to_excel(     writer, sheet_name="Ham Veri",           index=False)

print(f"✅ Excel kaydedildi: {CIKTI_DOSYASI}")
print("\n🎉 Tüm işlem tamamlandı! (Random Forest)")